In [ ]:
!pip install -q openai pandas numpy tqdm transformers torch sentence-transformers faiss-cpu

In [ ]:
import os, json, time, hashlib
import numpy as np
import pandas as pd
from getpass import getpass
from tqdm import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer
import faiss

from openai import AzureOpenAI
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
os.environ["AZURE_OPENAI_API_KEY"] = getpass("Paste AZURE_OPENAI_API_KEY: ").strip()
os.environ["ENDPOINT_URL"] = getpass("Paste ENDPOINT_URL: ").strip()
os.environ["API_VERSION"] = getpass("Paste API_VERSION: ").strip()

client = AzureOpenAI(
    azure_endpoint=os.getenv("ENDPOINT_URL"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version= os.getenv("API_VERSION"),
)

GPT_MODEL = "gpt-5-mini"

In [ ]:
# ============================================================
# PATHS
# ============================================================

RAW_DATA_PATH = "data/raw/comments_cmv.csv"

FINAL_META_DIR = "data/final_repositories"

TOPIC_META_FILES = {
    "blm": "blm_meta.parquet",
    "guns": "guns_meta.parquet",
    "samesex": "samesex_meta.parquet",
}

TEXT_COL = "comment_text"
STANCE_COL = "stance_label"

BASELINE_CACHE_DIR = "cache/mazare_dense_baseline_cache"
os.makedirs(BASELINE_CACHE_DIR, exist_ok=True)

RAW_META_CACHE = os.path.join(BASELINE_CACHE_DIR, "raw_meta_for_mazare.parquet")
RAW_EMB_CACHE = os.path.join(BASELINE_CACHE_DIR, "raw_mazare_embeddings.npy")
RAW_FAISS_CACHE = os.path.join(BASELINE_CACHE_DIR, "raw_mazare_faiss.index")

In [ ]:
# ============================================================
# Topic utilities
# ============================================================

TOPIC_NAMES = {
    "blm": "Black Lives Matter",
    "guns": "Gun control",
    "samesex": "Same-sex marriage legalization",
}

def topic_name(topic):
    return TOPIC_NAMES[topic]

def opposite_stance(stance):
    if stance == "pro":
        return "anti"
    if stance == "anti":
        return "pro"
    return None

In [ ]:
# ============================================================
# Load datasets
# ============================================================

def load_raw_dataset():
    raw = pd.read_csv(RAW_DATA_PATH, encoding="utf-8-sig")

    if "comment_text" not in raw.columns:
        if "message" in raw.columns:
            raw["comment_text"] = raw["message"]
        else:
            raise ValueError(f"Raw dataset needs comment_text or message. Columns: {raw.columns.tolist()}")

    if "topic_title" not in raw.columns:
        raw["topic_title"] = ""

    raw["comment_text"] = raw["comment_text"].fillna("").astype(str)
    raw["topic_title"] = raw["topic_title"].fillna("").astype(str)

    raw = raw[raw["comment_text"].str.len() > 30].copy()
    raw = raw.reset_index(drop=True)

    return raw


def load_final_meta(topic):
    path = os.path.join(FINAL_META_DIR, TOPIC_META_FILES[topic])

    if path.endswith(".parquet"):
        df = pd.read_parquet(path)
    else:
        df = pd.read_csv(path, encoding="utf-8-sig")

    if "comment_text" not in df.columns and "message" in df.columns:
        df["comment_text"] = df["message"]

    df[STANCE_COL] = df[STANCE_COL].astype(str).str.lower().str.strip()
    df = df[df[STANCE_COL].isin(["pro", "anti"])].copy()
    df = df.reset_index(drop=True)

    return df

In [ ]:
def call_gpt(
    prompt,
    system="You are a helpful assistant.",
    max_tokens=1200,
    retries=3,
    sleep_base=3,
):
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=GPT_MODEL,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": prompt},
                ],
                max_completion_tokens=max_tokens,
            )

            msg = resp.choices[0].message
            txt = (msg.content or "").strip()
            finish_reason = resp.choices[0].finish_reason

            print("finish_reason:", finish_reason)
            print("usage:", getattr(resp, "usage", None))

            if txt:
                return txt

            print("EMPTY GPT OUTPUT, attempt:", attempt + 1)

        except Exception as e:
            print("GPT ERROR:", repr(e))

        time.sleep(sleep_base * (attempt + 1))

    return ""

In [ ]:
def gpt_json(prompt, system="Return valid JSON only.", max_tokens=1600):
    txt = call_gpt(
        prompt=prompt,
        system=system + "\nReturn ONLY a valid JSON object. No markdown. No explanation.",
        max_tokens=max_tokens,
    )

    if not txt:
        return {"parse_error": True, "raw": ""}

    try:
        return json.loads(txt)
    except Exception:
        cleaned = txt.replace("```json", "").replace("```", "").strip()
        start = cleaned.find("{")
        end = cleaned.rfind("}")

        if start != -1 and end != -1 and end > start:
            try:
                return json.loads(cleaned[start:end + 1])
            except Exception:
                pass

    print("JSON parse failed. Raw text:", txt[:500])
    return {"parse_error": True, "raw": txt}

In [ ]:
# ============================================================
# Build/load Mazaré-style dense retrieval index
# ============================================================

DENSE_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
dense_model = SentenceTransformer(DENSE_MODEL_NAME)

def build_or_load_mazare_dense_index(force_rebuild=False):
    """
    Adaptation of Mazaré et al. (2018):
    Treat raw Reddit comments as a large persona memory and retrieve by semantic similarity.

    This baseline intentionally does NOT use:
    - topic extraction from our method
    - stance labels
    - NLI
    - GPT relevance filtering
    - balanced repositories
    - diversity selection
    - final 300 datasets
    """

    if (
        not force_rebuild
        and os.path.exists(RAW_META_CACHE)
        and os.path.exists(RAW_EMB_CACHE)
        and os.path.exists(RAW_FAISS_CACHE)
    ):
        print("Loading cached Mazaré-style dense retrieval artifacts...")
        raw_meta = pd.read_parquet(RAW_META_CACHE)
        embs = np.load(RAW_EMB_CACHE, mmap_mode="r")
        index = faiss.read_index(RAW_FAISS_CACHE)
        print("Loaded:", len(raw_meta), embs.shape, index.ntotal)
        return raw_meta, embs, index

    print("Building Mazaré-style dense retrieval index from raw dataset...")

    raw_meta = load_raw_dataset()

    keep_cols = []
    for c in ["comment_id", "author", "comment_text", "topic_title", "thread_id", "delta"]:
        if c in raw_meta.columns:
            keep_cols.append(c)

    raw_meta = raw_meta[keep_cols].copy()
    raw_meta = raw_meta.reset_index(drop=True)
    raw_meta.to_parquet(RAW_META_CACHE, index=False)

    texts = raw_meta["comment_text"].fillna("").astype(str).tolist()

    embs = dense_model.encode(
        texts,
        batch_size=256,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    np.save(RAW_EMB_CACHE, embs)

    dim = embs.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embs)

    faiss.write_index(index, RAW_FAISS_CACHE)

    print("Built index:", index.ntotal)

    return raw_meta, embs, index


raw_mazare_meta, raw_mazare_embs, raw_mazare_index = build_or_load_mazare_dense_index(
    force_rebuild=False
)

In [ ]:
def mazare_retrieve_candidates(user_opinion, topic, top_k_retrieve=60):
    query_text = f"{topic_name(topic)}. {user_opinion}"

    query_emb = dense_model.encode(
        [query_text],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    scores, ids = raw_mazare_index.search(query_emb, top_k_retrieve)

    retrieved = []

    for rank, idx in enumerate(ids[0]):
        idx = int(idx)
        row = raw_mazare_meta.iloc[idx].to_dict()

        retrieved.append({
            "rank": rank + 1,
            "similarity": float(scores[0][rank]),
            "text": str(row.get("comment_text", "")),
            "topic_title": str(row.get("topic_title", "")),
            "raw_index": idx,
        })

    return retrieved

In [ ]:
def safe_json_loads(txt):
    txt = (txt or "").strip()
    if not txt:
        return {"parse_error": True, "raw": ""}

    try:
        return json.loads(txt)
    except Exception:
        cleaned = txt.replace("```json", "").replace("```", "").strip()
        start = cleaned.find("{")
        end = cleaned.rfind("}")
        if start != -1 and end != -1 and end > start:
            try:
                return json.loads(cleaned[start:end + 1])
            except Exception:
                pass

    return {"parse_error": True, "raw": txt}


def call_gpt_quiet(prompt, system="You are a helpful assistant.", max_tokens=1800, retries=3):
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=GPT_MODEL,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": prompt},
                ],
                max_completion_tokens=max_tokens,
            )
            txt = (resp.choices[0].message.content or "").strip()
            if txt:
                return txt
        except Exception as e:
            print("GPT error:", repr(e))

        time.sleep(2 * (attempt + 1))

    return ""


def gpt_json_quiet(prompt, system="Return valid JSON only.", max_tokens=2200):
    txt = call_gpt_quiet(
        prompt=prompt,
        system=system + "\nReturn ONLY valid JSON. No markdown. No explanation.",
        max_tokens=max_tokens,
    )
    return safe_json_loads(txt)

In [ ]:
PERSONADB_DIR = "cache/personadb_offline_baseline"
os.makedirs(PERSONADB_DIR, exist_ok=True)

PERSONADB_PARQUET = os.path.join(PERSONADB_DIR, "personadb_refined_items.parquet")
PERSONADB_EMBS = os.path.join(PERSONADB_DIR, "personadb_refined_embeddings.npy")
PERSONADB_FAISS = os.path.join(PERSONADB_DIR, "personadb_refined_faiss.index")

DENSE_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
dense_model = SentenceTransformer(DENSE_MODEL_NAME)

In [ ]:
def make_personadb_user_groups(
    raw_df,
    user_col="author",
    text_col="comment_text",
    min_items_per_user=3,
    max_items_per_user=20,
    max_users=500,
    random_state=42,
):
    """
    Persona-DB assumes user histories.
    In your Reddit dataset, we approximate this by grouping comments by author.
    """

    df = raw_df.copy()

    if user_col not in df.columns:
        df[user_col] = "unknown_user"

    df[text_col] = df[text_col].fillna("").astype(str)
    df = df[df[text_col].str.len() > 30].copy()

    counts = df[user_col].value_counts()
    valid_users = counts[counts >= min_items_per_user].index.tolist()

    if max_users is not None:
        rng = np.random.default_rng(random_state)
        valid_users = list(rng.choice(valid_users, size=min(max_users, len(valid_users)), replace=False))

    groups = []

    for uid in valid_users:
        user_rows = df[df[user_col] == uid].copy()
        if len(user_rows) > max_items_per_user:
            user_rows = user_rows.sample(max_items_per_user, random_state=random_state)

        history = user_rows[text_col].astype(str).tolist()

        groups.append({
            "user_id": str(uid),
            "history": history,
        })

    return groups

In [ ]:
def personadb_offline_extract_user_database(user_id, history):
    """
    Offline hierarchy construction:
    History -> Distilled Persona -> Induced Persona -> Cache
    """

    compact_history = [
        {"id": str(i), "text": txt[:500]}
        for i, txt in enumerate(history)
        if str(txt).strip()
    ]

    prompt = f"""
You are constructing a Persona-DB style hierarchical persona database.

User id:
{user_id}

User history:
{json.dumps(compact_history, ensure_ascii=False)}

Task:
Create a compact persona database from this user's history.

Return a valid JSON object with this exact schema:
{{
  "distilled_personas": [
    {{
      "key": "short concrete label",
      "description": "specific opinion, attitude, preference, or communication pattern grounded in the user's history",
      "supporting_history_ids": ["0", "2"]
    }}
  ],
  "induced_personas": [
    {{
      "key": "short abstract label",
      "description": "higher-level inferred value, worldview, concern, or generalizable trait",
      "source_distilled_keys": ["key1", "key2"]
    }}
  ],
  "cache_keys": [
    "short reusable category",
    "another reusable category"
  ]
}}

Guidelines:
- Distilled personas should be concrete and grounded.
- Induced personas should be higher-level and generalizable.
- Do not invent unsupported facts.
- Do not create contradiction labels.
- Do not classify pro/anti.
- Return 3-8 distilled personas, 2-6 induced personas, and 3-8 cache keys.
"""

    data = gpt_json_quiet(
        prompt,
        system="You build concise Persona-DB hierarchical user persona databases.",
        max_tokens=2400,
    )

    return data

In [ ]:
def build_personadb_offline(
    raw_df,
    force_rebuild=False,
    max_users=500,
    min_items_per_user=3,
    max_items_per_user=20,
):
    """
    Builds and saves the offline Persona-DB refined database.
    Run this once.
    """

    if (not force_rebuild) and os.path.exists(PERSONADB_PARQUET):
        print("Loading existing Persona-DB refined database...")
        return pd.read_parquet(PERSONADB_PARQUET)

    groups = make_personadb_user_groups(
        raw_df=raw_df,
        user_col="author",
        text_col="comment_text",
        min_items_per_user=min_items_per_user,
        max_items_per_user=max_items_per_user,
        max_users=max_users,
    )

    print("Number of user groups:", len(groups))

    rows = []

    for g in tqdm(groups):
        user_id = g["user_id"]
        history = g["history"]

        data = personadb_offline_extract_user_database(user_id, history)

        if data.get("parse_error"):
            continue

        distilled = data.get("distilled_personas", [])
        induced = data.get("induced_personas", [])
        cache_keys = data.get("cache_keys", [])

        for i, dp in enumerate(distilled):
            rows.append({
                "user_id": user_id,
                "layer": "distilled_persona",
                "key": str(dp.get("key", "")),
                "description": str(dp.get("description", "")),
                "cache_keys": json.dumps(cache_keys, ensure_ascii=False),
                "source": "offline_personadb",
            })

        for i, ip in enumerate(induced):
            rows.append({
                "user_id": user_id,
                "layer": "induced_persona",
                "key": str(ip.get("key", "")),
                "description": str(ip.get("description", "")),
                "cache_keys": json.dumps(cache_keys, ensure_ascii=False),
                "source": "offline_personadb",
            })

    personadb_df = pd.DataFrame(rows)
    personadb_df = personadb_df[personadb_df["description"].astype(str).str.len() > 10].copy()
    personadb_df = personadb_df.reset_index(drop=True)

    personadb_df.to_parquet(PERSONADB_PARQUET, index=False)
    print("Saved:", PERSONADB_PARQUET)
    print("Rows:", len(personadb_df))

    return personadb_df

In [ ]:
# raw_df should be your original 700K Reddit dataframe
# If you already loaded it as raw_mazare_meta, you can use that.
personadb_df = build_personadb_offline(
    raw_df=raw_mazare_meta,
    force_rebuild=False,
    max_users=500,          # increase later if you want stronger baseline
    min_items_per_user=3,
    max_items_per_user=20,
)

In [ ]:
def build_or_load_personadb_index(force_rebuild=False):
    if (
        not force_rebuild
        and os.path.exists(PERSONADB_PARQUET)
        and os.path.exists(PERSONADB_EMBS)
        and os.path.exists(PERSONADB_FAISS)
    ):
        print("Loading Persona-DB index...")
        df = pd.read_parquet(PERSONADB_PARQUET)
        embs = np.load(PERSONADB_EMBS, mmap_mode="r")
        index = faiss.read_index(PERSONADB_FAISS)
        print("Loaded:", len(df), embs.shape, index.ntotal)
        return df, embs, index

    df = pd.read_parquet(PERSONADB_PARQUET)

    texts = (
        df["layer"].astype(str)
        + ": "
        + df["key"].astype(str)
        + ". "
        + df["description"].astype(str)
    ).tolist()

    embs = dense_model.encode(
        texts,
        batch_size=256,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    np.save(PERSONADB_EMBS, embs)

    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)

    faiss.write_index(index, PERSONADB_FAISS)

    print("Saved Persona-DB index.")
    return df, embs, index

In [ ]:
personadb_df, personadb_embs, personadb_index = build_or_load_personadb_index(
    force_rebuild=False
)

In [ ]:
# ============================================================
# Mazaré baseline 1: retrieval only
# No GPT decision.
# This is the cleanest test of semantic persona retrieval.
# ============================================================

def method_mazare_retrieval_only(
    user_opinion,
    topic,
    expected_user_stance,
    top_k_retrieve=60,
):
    retrieved = mazare_retrieve_candidates(
        user_opinion=user_opinion,
        topic=topic,
        top_k_retrieve=top_k_retrieve,
    )

    if not retrieved:
        return ""

    return retrieved[0]["text"]

In [ ]:
# ============================================================
# Mazaré baseline 2: retrieval + GPT generation
# Important: GPT is NOT part of the original paper.
# This is only an adaptation for generating a comparable persona output.
# We do NOT give GPT the target opposite stance.
# ============================================================

def method_mazare_retrieval_plus_gpt_no_target_stance(
    user_opinion,
    topic,
    expected_user_stance,
    top_k_retrieve=60,
    top_k_persona=8,
):
    retrieved = mazare_retrieve_candidates(
        user_opinion=user_opinion,
        topic=topic,
        top_k_retrieve=top_k_retrieve,
    )

    persona_evidence = [
        {
            "rank": r["rank"],
            "similarity": r["similarity"],
            "topic_title": r["topic_title"][:150],
            "text": r["text"][:700],
        }
        for r in retrieved[:top_k_persona]
    ]

    prompt = f"""
Topic:
{topic_name(topic)}

User opinion:
{user_opinion}

Retrieved persona evidence from a large raw Reddit dataset:
{json.dumps(persona_evidence, ensure_ascii=False)}

Task:
Write one short Reddit-style persona comment grounded in the retrieved evidence.
Do not force an opposite stance unless the retrieved evidence supports it.
Do not mention retrieval, dataset, or AI.
Write 2-4 sentences only.
"""

    out = call_gpt(
        prompt,
        system="You generate concise persona-grounded Reddit-style comments from retrieved evidence.",
        max_tokens=1200,
    )

    # if not out and persona_evidence:
    #     out = persona_evidence[0]["text"]

    return out

In [ ]:
# ============================================================
# Partner Matters baseline:
# persona fusion for retrieval-based response selection
# inspired by Gu et al. (SIGIR 2021)
# ============================================================

def cosine(a, b):
    return float(np.dot(a, b))


def partner_matters_retrieve_personas(
    user_opinion,
    topic,
    top_k_personas=8,
):
    """
    Retrieve persona candidates from the raw 700K semantic index.
    This is NOT stance-aware and does NOT use NLI.
    """

    query_text = f"{topic_name(topic)}. {user_opinion}"

    query_emb = dense_model.encode(
        [query_text],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    scores, ids = raw_mazare_index.search(query_emb, top_k_personas)

    personas = []

    for rank, idx in enumerate(ids[0]):
        idx = int(idx)
        row = raw_mazare_meta.iloc[idx].to_dict()

        personas.append({
            "rank": rank + 1,
            "similarity": float(scores[0][rank]),
            "text": str(row.get("comment_text", ""))[:900],
            "topic_title": str(row.get("topic_title", ""))[:200],
        })

    return personas


def partner_matters_build_response_candidates(
    user_opinion,
    topic,
    expected_user_stance,
    raw_top_k=60,
    candidate_k=12,
):
    """
    Build response candidates from raw semantic retrieval.
    In the original paper, response candidates are given by the response-selection dataset.
    In our adaptation, we use retrieved raw Reddit comments as candidate responses.
    """

    retrieved = mazare_retrieve_candidates(
        user_opinion=user_opinion,
        topic=topic,
        top_k_retrieve=raw_top_k,
    )

    candidates = []

    for r in retrieved[:candidate_k]:
        txt = str(r["text"]).strip()
        if txt:
            candidates.append({
                "text": txt[:900],
                "similarity": float(r["similarity"]),
                "topic_title": str(r.get("topic_title", ""))[:200],
            })

    return candidates

In [ ]:
def method_partner_matters_persona_fusion_response_selection(
    user_opinion,
    topic,
    expected_user_stance,
    top_k_personas=8,
    raw_top_k_candidates=60,
    candidate_k=12,
):
    """
    Adaptation of:
    Gu et al. 2021, Partner Matters!

    Core idea:
    - retrieve persona evidence
    - retrieve candidate responses
    - fuse context/persona/response signals
    - select the response with the best fused score

    No GPT.
    No NLI.
    No stance labels.
    No final 300 repository.
    """

    target_stance = opposite_stance(expected_user_stance)

    # Context embedding
    context_text = f"{topic_name(topic)}. {user_opinion}"
    context_emb = dense_model.encode(
        [context_text],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")[0]

    # Retrieve multiple persona candidates
    personas = partner_matters_retrieve_personas(
        user_opinion=user_opinion,
        topic=topic,
        top_k_personas=top_k_personas,
    )

    persona_texts = [p["text"] for p in personas if str(p["text"]).strip()]

    if not persona_texts:
        return ""

    persona_embs = dense_model.encode(
        persona_texts,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    # Persona fusion: average retrieved persona representations
    fused_persona_emb = persona_embs.mean(axis=0)
    fused_persona_emb = fused_persona_emb / (np.linalg.norm(fused_persona_emb) + 1e-12)

    # Retrieve candidate responses from raw dataset
    response_candidates = partner_matters_build_response_candidates(
        user_opinion=user_opinion,
        topic=topic,
        expected_user_stance=expected_user_stance,
        raw_top_k=raw_top_k_candidates,
        candidate_k=candidate_k,
    )

    if not response_candidates:
        return persona_texts[0]

    response_texts = [c["text"] for c in response_candidates]

    response_embs = dense_model.encode(
        response_texts,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    scored = []

    for i, c in enumerate(response_candidates):
        r_emb = response_embs[i]

        # context-response matching
        context_response_score = cosine(context_emb, r_emb)

        # persona-response matching
        persona_response_score = cosine(fused_persona_emb, r_emb)

        # context-persona matching
        context_persona_score = cosine(context_emb, fused_persona_emb)

        # Fusion score inspired by context/persona/response interaction
        fusion_score = (
            0.45 * context_response_score
            + 0.45 * persona_response_score
            + 0.10 * context_persona_score
        )

        scored.append({
            "text": c["text"],
            "fusion_score": fusion_score,
            "context_response_score": context_response_score,
            "persona_response_score": persona_response_score,
            "context_persona_score": context_persona_score,
        })

    best = max(scored, key=lambda x: x["fusion_score"])

    return best["text"]

In [ ]:
def coo_retrieve_semantic_candidates(
    user_opinion,
    topic,
    top_k_retrieve=60,
):
    """
    Retrieves semantic candidates from the raw 700K FAISS index.
    This reuses the same raw_mazare_index and raw_mazare_meta
    that were already built for the Mazaré baseline.
    """

    query_text = f"{topic_name(topic)}. {user_opinion}"

    query_emb = dense_model.encode(
        [query_text],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    scores, ids = raw_mazare_index.search(query_emb, top_k_retrieve)

    candidates = []

    for rank, idx in enumerate(ids[0]):
        idx = int(idx)
        row = raw_mazare_meta.iloc[idx].to_dict()

        txt = str(row.get("comment_text", "")).strip()

        if not txt:
            continue

        candidates.append({
            "id": str(rank),
            "raw_index": idx,
            "semantic_rank": rank + 1,
            "semantic_similarity": float(scores[0][rank]),
            "topic_title": str(row.get("topic_title", ""))[:200],
            "text": txt[:900],
        })

    return candidates

In [ ]:
def coo_style_rank_by_usefulness(
    user_opinion,
    topic,
    candidates,
    top_k_useful=8,
):
    """
    COO-style LLMtop-K adaptation.

    In the COO paper, the LLM ranks implicit personae/historical opinions
    by usefulness for predicting the user's opinion.

    Here, we adapt that idea:
    The LLM ranks raw retrieved comments by usefulness for constructing
    a relevant persona on the given topic.

    It is intentionally NOT asked to rank by contradiction.
    """

    if not candidates:
        return []

    compact_candidates = [
        {
            "id": c["id"],
            "semantic_rank": c["semantic_rank"],
            "semantic_similarity": round(c["semantic_similarity"], 4),
            "topic_title": c["topic_title"],
            "text": c["text"][:700],
        }
        for c in candidates
    ]

    prompt = f"""
You are ranking candidate persona evidence for a personalized dialogue system.

Topic:
{topic_name(topic)}

User opinion:
{user_opinion}

Candidate comments retrieved from a large raw Reddit dataset:
{json.dumps(compact_candidates, ensure_ascii=False)}

Task:
Rank the candidate comments by how useful they are as persona evidence
for constructing a realistic Reddit-style persona related to the topic.

Important:
- Rank by usefulness, relevance, clarity, and persona richness.
- Do NOT rank by contradiction.
- Do NOT assume the selected persona must oppose the user.
- Ignore irrelevant, unclear, off-topic, or low-quality comments.

Return a valid JSON object with this exact schema:
{{
  "ranked_ids": ["id1", "id2", "id3"],
  "reasons": {{
    "id1": "brief reason",
    "id2": "brief reason"
  }}
}}

Return at most {top_k_useful} ids.
Do not output any text outside the JSON object.
"""

    data = gpt_json(
        prompt,
        system="You are a careful persona-evidence ranker. Return valid JSON only.",
        max_tokens=1200,
    )

    ranked_ids = data.get("ranked_ids", [])

    if not isinstance(ranked_ids, list):
        ranked_ids = []

    ranked_ids = [str(x) for x in ranked_ids]

    by_id = {str(c["id"]): c for c in candidates}

    selected = []

    for rid in ranked_ids:
        if rid in by_id:
            selected.append(by_id[rid])
        if len(selected) >= top_k_useful:
            break

    # fallback: if JSON failed or ids are bad, use semantic top-k
    if not selected:
        selected = candidates[:top_k_useful]

    return selected

In [ ]:
def method_coo_style_usefulness_ranking(
    user_opinion,
    topic,
    expected_user_stance,
    top_k_retrieve=60,
    top_k_useful=8,
):
    """
    COO-style usefulness baseline.

    Pipeline:
    1. Retrieve top-60 semantic candidates from raw 700K dataset.
    2. Ask GPT to rank them by usefulness as persona evidence.
    3. Use top-8 useful candidates.
    4. Generate a persona from those selected candidates.

    This baseline tests:
    Is usefulness-aware persona selection enough,
    without explicit contradiction modeling?
    """

    target_stance = opposite_stance(expected_user_stance)

    candidates = coo_retrieve_semantic_candidates(
        user_opinion=user_opinion,
        topic=topic,
        top_k_retrieve=top_k_retrieve,
    )

    selected = coo_style_rank_by_usefulness(
        user_opinion=user_opinion,
        topic=topic,
        candidates=candidates,
        top_k_useful=top_k_useful,
    )

    selected_compact = [
        {
            "id": c["id"],
            "semantic_rank": c["semantic_rank"],
            "semantic_similarity": round(c["semantic_similarity"], 4),
            "topic_title": c["topic_title"],
            "text": c["text"][:700],
        }
        for c in selected
    ]

    prompt = f"""
You are constructing a Reddit-style persona comment from selected useful persona evidence.

Topic:
{topic_name(topic)}

User opinion:
{user_opinion}

Selected useful persona evidence:
{json.dumps(selected_compact, ensure_ascii=False)}

Task:
Write one short Reddit-style persona comment grounded in the selected evidence.

Important:
- Use the selected evidence for style, topic framing, and persona realism.
- Do not copy the evidence exactly.
- Do not mention the dataset, retrieval, or AI.
- Write 2-4 sentences only.

Note:
This baseline ranks by usefulness, not contradiction.
Therefore, do not force an opposite stance unless the selected evidence clearly supports it.
"""

    out = call_gpt(
        prompt,
        system="You generate concise persona-grounded Reddit-style comments from selected useful evidence.",
        max_tokens=1200,
    )

    # if not out and selected:
    #     out = selected[0]["text"]

    # if not out:
    #     out = ""

    return out

In [ ]:
def personadb_online_retrieve(
    user_opinion,
    topic,
    top_k_items=8,
):
    """
    Online Persona-DB retrieval:
    retrieve from the precomputed refined persona database.
    """

    query = f"{topic_name(topic)}. {user_opinion}"

    q_emb = dense_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    scores, ids = personadb_index.search(q_emb, top_k_items)

    retrieved = []

    for rank, idx in enumerate(ids[0]):
        idx = int(idx)
        row = personadb_df.iloc[idx].to_dict()

        retrieved.append({
            "rank": rank + 1,
            "score": float(scores[0][rank]),
            "user_id": str(row.get("user_id", "")),
            "layer": str(row.get("layer", "")),
            "key": str(row.get("key", "")),
            "description": str(row.get("description", "")),
        })

    return retrieved

In [ ]:
def method_personadb_offline_online_baseline(
    user_opinion,
    topic,
    expected_user_stance,
    top_k_items=8,
):
    """
    Fair Persona-DB-style baseline:
    Offline hierarchy already exists.
    Online stage only retrieves refined persona items and generates output.

    No stance filtering.
    No NLI.
    No contradiction reranking.
    No balanced pro/anti repository.
    """

    retrieved_items = personadb_online_retrieve(
        user_opinion=user_opinion,
        topic=topic,
        top_k_items=top_k_items,
    )

    if not retrieved_items:
        return ""

    prompt = f"""
You are generating a Reddit-style persona comment from a precomputed Persona-DB refined persona database.

Topic:
{topic_name(topic)}

User opinion:
{user_opinion}

Retrieved refined persona items:
{json.dumps(retrieved_items, ensure_ascii=False)}

Task:
Write one short Reddit-style persona comment grounded in the retrieved refined persona items.

Important:
- Use the retrieved items for worldview, style, concerns, and topic framing.
- Do not mention Persona-DB, retrieval, dataset, or AI.
- Do not copy the items exactly.
- Write 2-4 sentences only.
- This baseline is not contradiction-aware, so do not force an opposite stance unless the retrieved items naturally support it.
"""

    out = call_gpt_quiet(
        prompt,
        system="You generate concise Reddit-style persona comments from retrieved refined persona items.",
        max_tokens=1000,
    )

    return out

In [ ]:
# ============================================================
# Our method
# ============================================================

def runtime_pool_score(df):
    df = df.copy()

    for col in [
        "gpt_final_score",
        "keep_score",
        "persona_usefulness",
        "coverage_score",
        "uniqueness_score",
        "clarity_score",
        "redundancy_risk",
        "hnsw_similarity",
        "topic_candidate_score",
    ]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

    score = np.zeros(len(df), dtype=np.float32)

    if "gpt_final_score" in df.columns:
        score += 2.0 * df["gpt_final_score"].values
    if "keep_score" in df.columns:
        score += 1.5 * df["keep_score"].values
    if "persona_usefulness" in df.columns:
        score += 1.2 * df["persona_usefulness"].values
    if "coverage_score" in df.columns:
        score += 1.0 * df["coverage_score"].values
    if "uniqueness_score" in df.columns:
        score += 0.8 * df["uniqueness_score"].values
    if "clarity_score" in df.columns:
        score += 0.5 * df["clarity_score"].values
    if "redundancy_risk" in df.columns:
        score -= 0.8 * df["redundancy_risk"].values
    if "hnsw_similarity" in df.columns:
        score += 0.5 * df["hnsw_similarity"].values
    if "topic_candidate_score" in df.columns:
        score += 0.3 * df["topic_candidate_score"].values

    return score


def method_ours_nli_gpt(
    user_opinion,
    topic,
    expected_user_stance,
    meta,
    pool_size=60,
    k2_short=8,
):
    target_stance = opposite_stance(expected_user_stance)

    pool = meta[meta[STANCE_COL] == target_stance].copy()
    pool = pool.reset_index(drop=True)

    if pool.empty:
        return ""

    pool["_runtime_pool_score"] = runtime_pool_score(pool)
    pool = pool.sort_values("_runtime_pool_score", ascending=False).head(pool_size).copy()

    texts = pool[TEXT_COL].fillna("").astype(str).tolist()

    nli_scores = method_nli_contradiction_scores_batch(
        premise=user_opinion,
        hypotheses=texts,
        batch_size=16,
    )

    pool["nli_score"] = nli_scores
    shortlist = pool.sort_values("nli_score", ascending=False).head(k2_short).copy()

    candidates = []
    for i, row in shortlist.iterrows():
        candidates.append({
            "id": str(i),
            "text": str(row[TEXT_COL])[:700],
            "nli_score": float(row["nli_score"]),
        })

    if not candidates:
        return ""

    prompt = f"""
Topic: {topic_name(topic)}

User opinion:
{user_opinion}

Candidate opposite-stance comments:
{json.dumps(candidates, ensure_ascii=False)}

Select the single best comment for constructing an opposite-opinion persona.
Prefer direct contradiction, topic relevance, clarity, and persona usefulness.

Return a valid JSON object with this exact schema:
{{
  "best_id": "id",
  "reason": "brief reason"
}}
Do not output any text outside the JSON object.
"""

    data = gpt_json(
        prompt,
        system="You are a careful contradiction reranker. Return valid JSON only.",
        max_tokens=1200,
    )

    best_id = str(data.get("best_id", candidates[0]["id"]))
    best = next((c for c in candidates if c["id"] == best_id), candidates[0])

    return best["text"]

In [ ]:
# ============================================================
# NLI models
# IMPORTANT:
# - METHOD NLI is used inside our method for NLI reranking.
# - EVALUATION NLI is used only for the reported NLI metric.
# This avoids evaluating our method with the exact same NLI model
# used inside the method.
# ============================================================

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

METHOD_NLI_MODEL_NAME = "MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli"
EVAL_NLI_MODEL_NAME = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"

# Method NLI model: used inside our retrieval/reranking pipeline
method_tokenizer = AutoTokenizer.from_pretrained(METHOD_NLI_MODEL_NAME)
method_nli_model = AutoModelForSequenceClassification.from_pretrained(
    METHOD_NLI_MODEL_NAME
).to(device)
method_nli_model.eval()

# Evaluation NLI model: used only for the reported metric
eval_tokenizer = AutoTokenizer.from_pretrained(EVAL_NLI_MODEL_NAME)
eval_nli_model = AutoModelForSequenceClassification.from_pretrained(
    EVAL_NLI_MODEL_NAME
).to(device)
eval_nli_model.eval()

print("Device:", device)
print("Method NLI model:", METHOD_NLI_MODEL_NAME)
print("Evaluation NLI model:", EVAL_NLI_MODEL_NAME)

In [ ]:
# ============================================================
# Generic NLI contradiction helpers
# ============================================================

def find_contradiction_index(model):
    """
    Finds the contradiction label index for any HuggingFace NLI model.
    """

    id2label = model.config.id2label

    for idx, label in id2label.items():
        if "contradiction" in str(label).lower():
            return int(idx)

    raise ValueError(f"Could not find contradiction label in: {id2label}")


METHOD_CONTRA_IDX = find_contradiction_index(method_nli_model)
EVAL_CONTRA_IDX = find_contradiction_index(eval_nli_model)

print("Method contradiction index:", METHOD_CONTRA_IDX)
print("Evaluation contradiction index:", EVAL_CONTRA_IDX)


@torch.inference_mode()
def get_contradiction_score_with_model(
    premise,
    hypothesis,
    tokenizer,
    model,
    contradiction_idx,
    max_length=512,
):
    """
    Computes contradiction probability for one premise-hypothesis pair.
    """

    if not str(hypothesis).strip():
        return None

    batch = tokenizer(
        str(premise),
        str(hypothesis),
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_length,
    )

    batch = {k: v.to(device) for k, v in batch.items()}

    logits = model(**batch).logits
    probs = torch.softmax(logits, dim=-1)[0]

    return float(probs[contradiction_idx].detach().cpu())


@torch.inference_mode()
def get_contradiction_scores_batch_with_model(
    premise,
    hypotheses,
    tokenizer,
    model,
    contradiction_idx,
    batch_size=16,
    max_length=512,
):
    """
    Computes contradiction probabilities for one premise and many hypotheses.
    """

    scores = []

    for i in range(0, len(hypotheses), batch_size):
        batch_h = hypotheses[i:i + batch_size]
        premises = [str(premise)] * len(batch_h)

        batch = tokenizer(
            premises,
            [str(h) for h in batch_h],
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_length,
        )

        batch = {k: v.to(device) for k, v in batch.items()}

        logits = model(**batch).logits
        probs = torch.softmax(logits, dim=-1)[:, contradiction_idx]

        scores.extend(probs.detach().cpu().numpy().tolist())

    return np.array(scores, dtype=np.float32)

In [ ]:
# ============================================================
# Method NLI functions
# Used ONLY inside our method for retrieval/reranking
# ============================================================

def method_nli_contradiction_score(premise, hypothesis):
    return get_contradiction_score_with_model(
        premise=premise,
        hypothesis=hypothesis,
        tokenizer=method_tokenizer,
        model=method_nli_model,
        contradiction_idx=METHOD_CONTRA_IDX,
    )


def method_nli_contradiction_scores_batch(premise, hypotheses, batch_size=16):
    return get_contradiction_scores_batch_with_model(
        premise=premise,
        hypotheses=hypotheses,
        tokenizer=method_tokenizer,
        model=method_nli_model,
        contradiction_idx=METHOD_CONTRA_IDX,
        batch_size=batch_size,
    )

In [ ]:
# ============================================================
# Evaluation NLI functions
# Used ONLY for reported NLI metric
# ============================================================

def eval_nli_contradiction_score(premise, hypothesis):
    return get_contradiction_score_with_model(
        premise=premise,
        hypothesis=hypothesis,
        tokenizer=eval_tokenizer,
        model=eval_nli_model,
        contradiction_idx=EVAL_CONTRA_IDX,
    )


def eval_nli_contradiction_scores_batch(premise, hypotheses, batch_size=16):
    return get_contradiction_scores_batch_with_model(
        premise=premise,
        hypotheses=hypotheses,
        tokenizer=eval_tokenizer,
        model=eval_nli_model,
        contradiction_idx=EVAL_CONTRA_IDX,
        batch_size=batch_size,
    )

In [ ]:
# ============================================================
# Evaluation metrics used in the paper
# ============================================================

# ------------------------------------------------------------
# GPT stance classifier + opposite stance accuracy
# ------------------------------------------------------------

def classify_persona_stance_with_gpt(output_text, topic):
    if not str(output_text).strip():
        return {
            "predicted_persona_stance": "empty",
            "stance_reason": "empty output",
        }

    prompt = f"""
Topic:
{topic_name(topic)}

Persona text:
{output_text}

Classify the persona text's stance toward the topic.

Labels:
- "pro": supports the topic/movement/policy/right.
- "anti": opposes the topic/movement/policy/right.
- "unclear": unclear, mixed, irrelevant, or ambiguous.

Return valid JSON only:
{{
  "stance": "pro",
  "reason": "brief explanation"
}}
"""

    data = gpt_json_quiet(
        prompt,
        system="You are a careful stance classifier. Return valid JSON only.",
        max_tokens=800,
    )

    stance = str(data.get("stance", "unclear")).lower().strip()
    if stance not in ["pro", "anti", "unclear"]:
        stance = "unclear"

    return {
        "predicted_persona_stance": stance,
        "stance_reason": str(data.get("reason", ""))[:300],
    }


def compute_stance_accuracy(output_text, topic, expected_user_stance):
    target = opposite_stance(expected_user_stance)

    stance_info = classify_persona_stance_with_gpt(
        output_text=output_text,
        topic=topic,
    )

    predicted = stance_info["predicted_persona_stance"]

    return {
        **stance_info,
        "target_persona_stance": target,
        "opposite_stance_correct": bool(predicted == target),
        "opposite_stance_accuracy": 1.0 if predicted == target else 0.0,
    }


def evaluate_output(user_opinion, output_text, topic, expected_user_stance):
    has_text = bool(str(output_text).strip())

    nli_score = (
        eval_nli_contradiction_score(user_opinion, output_text)
        if has_text
        else None
    )

    stance_metrics = compute_stance_accuracy(
        output_text=output_text,
        topic=topic,
        expected_user_stance=expected_user_stance,
    )

    return {
        "eval_nli_model": EVAL_NLI_MODEL_NAME,
        "method_nli_model": METHOD_NLI_MODEL_NAME,
        "nli_contradiction": nli_score,
        **stance_metrics,
    }


In [ ]:
# ============================================================
# 1. Expanded balanced test set: 180 cases
# 3 topics × 2 stances × 30 examples
# ============================================================

def make_cases(topic, stance, opinions):
    return [
        {
            "topic": topic,
            "user_opinion": opinion,
            "expected_user_stance": stance,
        }
        for opinion in opinions
    ]


BLM_PRO = [
    "Black Lives Matter is important because police violence and racial discrimination are real problems that need public attention.",
    "I support BLM because Black communities still face systemic racism, and ignoring that only protects the status quo.",
    "BLM is necessary because racial injustice did not disappear, and people need to protest when institutions fail them.",
    "The movement matters because police accountability and racial equality should not be controversial ideas.",
    "I think BLM helped expose problems that many people preferred to ignore, especially around policing and discrimination.",
    "BLM gives people a way to demand justice when official institutions do not take police violence seriously.",
    "I support BLM because equal treatment under the law is still not experienced equally by everyone.",
    "The protests were justified because people were tired of seeing repeated cases of police brutality without accountability.",
    "BLM is not about hating anyone; it is about saying that Black lives should be protected and valued equally.",
    "I think the movement is important because racial inequality still affects policing, housing, education, and opportunity.",
    "BLM brought attention to injustice that many communities have been talking about for decades.",
    "Supporting BLM means supporting accountability, fairness, and equal protection for Black citizens.",
    "I believe BLM is needed because silence about racism allows unfair systems to continue.",
    "The movement helped make police violence and systemic discrimination part of the national conversation.",
    "I support BLM because people should not have to accept discrimination as normal.",
    "BLM is a legitimate response to real cases where Black people were treated unfairly or violently.",
    "I think criticizing BLM ignores the deeper problem of racial injustice that the movement is trying to address.",
    "The slogan Black Lives Matter is not saying other lives do not matter; it is highlighting a group that has been harmed.",
    "BLM is important because public pressure is often necessary before institutions make meaningful reforms.",
    "I support the movement because everyone should care when a community repeatedly faces unequal treatment.",
    "BLM represents a demand for dignity, fairness, and accountability in policing.",
    "I think the movement gave a voice to people who felt ignored by courts, police departments, and politicians.",
    "BLM is necessary because racism can exist in systems even when individuals deny being racist.",
    "The movement matters because police accountability should be a basic democratic principle.",
    "I support BLM because racial justice is not a radical demand.",
    "BLM helped show that many people experience law enforcement differently depending on race.",
    "I think the movement is justified because injustice should be confronted publicly, not hidden.",
    "BLM is important because communities need protection from both crime and abusive policing.",
    "I support BLM because equality requires recognizing when one group faces specific harms.",
    "The movement is necessary because without protest, many cases of injustice would be forgotten.",
]

BLM_ANTI = [
    "BLM is nonsense and only creates division between people who should be treated equally.",
    "I do not support BLM because I think it exaggerates racism and turns every problem into a racial issue.",
    "The movement caused more anger than solutions, and I do not think it actually helped society.",
    "I think BLM unfairly attacks police and ignores personal responsibility in communities.",
    "BLM seems more like political activism than a real solution to crime or social problems.",
    "I oppose BLM because it makes people focus on race instead of treating everyone as individuals.",
    "I think the movement blames police too quickly and ignores how dangerous their job can be.",
    "BLM has become too political and divisive to be taken seriously as a justice movement.",
    "I do not support BLM because I believe all lives matter and no group should be singled out.",
    "The movement often excuses riots and disorder instead of promoting real solutions.",
    "I think BLM creates hostility toward law enforcement and makes communities less safe.",
    "BLM focuses too much on systemic racism and not enough on personal responsibility and crime.",
    "I oppose the movement because it treats America as racist even though the country has made progress.",
    "I think BLM activists often ignore facts when a police shooting does not fit their narrative.",
    "The movement divides people into victims and oppressors instead of encouraging unity.",
    "I do not think BLM has improved race relations; it has made them worse.",
    "BLM became a political slogan rather than a practical plan for helping communities.",
    "I think the movement unfairly assumes police are racist before evidence is available.",
    "BLM ignores violence inside communities while blaming police for everything.",
    "I oppose BLM because its message often turns into anti-police rhetoric.",
    "The movement pressures people to agree with it or be labeled racist.",
    "I think BLM makes honest discussion about crime and policing almost impossible.",
    "BLM is not the right way to solve inequality because it encourages resentment.",
    "I do not support BLM because I think its leaders and activists push a political agenda.",
    "The movement treats disagreement as racism, which shuts down reasonable debate.",
    "I think BLM overstates the role of race in police encounters.",
    "BLM protests often caused chaos and damaged the communities they claimed to defend.",
    "I oppose BLM because public safety depends on supporting police, not demonizing them.",
    "I think the movement is based more on emotion than balanced evidence.",
    "BLM has become a symbol of division rather than equality.",
]

GUNS_PRO = [
    "Gun control is important because too many people can get dangerous weapons too easily.",
    "I support stricter gun laws because mass shootings and gun violence show that the current system is not working.",
    "There should be stronger background checks and limits on certain weapons because public safety matters more than unlimited access.",
    "Gun control does not mean banning everything; it means making sure irresponsible or dangerous people cannot easily get firearms.",
    "The government should regulate guns more strictly because the amount of gun violence in America is unacceptable.",
    "I support gun control because no right should be completely unlimited when public safety is at risk.",
    "Stronger gun laws are necessary because thoughts and prayers have not stopped mass shootings.",
    "I think background checks should be universal and enforced consistently across all states.",
    "Gun control is common sense when children have to practice active shooter drills in school.",
    "I support restrictions on assault-style weapons because civilians do not need military-level firepower.",
    "The Second Amendment should not prevent reasonable laws that reduce gun deaths.",
    "I think responsible gun ownership can exist together with stronger safety regulations.",
    "Gun violence is a public health problem, and policy should treat it seriously.",
    "I support waiting periods because impulsive violence and suicide can be reduced by slowing access.",
    "There should be stricter rules for people with violent histories or serious warning signs.",
    "I think America has too many guns and too few safeguards.",
    "Gun control is needed because easy access to firearms makes conflicts more deadly.",
    "I support licensing and training requirements because guns are dangerous tools.",
    "The current system prioritizes gun access over the safety of ordinary people.",
    "I think stricter gun laws would save lives without preventing responsible ownership.",
    "Mass shootings show that voluntary responsibility is not enough.",
    "I support banning high-capacity magazines because they make shootings deadlier.",
    "Gun control is necessary because other developed countries have far fewer gun deaths.",
    "I believe public safety should come before the gun lobby.",
    "Background checks should be stronger because dangerous people should not be able to buy guns easily.",
    "I support gun regulation because freedom also means being safe in schools, stores, and public places.",
    "The right to own guns should come with serious responsibilities and limits.",
    "I think gun laws should be updated to match the reality of modern weapons.",
    "Gun violence affects families every day, not only when mass shootings make the news.",
    "I support stricter gun control because preventable deaths should not be treated as normal.",
]

GUNS_ANTI = [
    "Gun control is a bad idea because people have the right to defend themselves and the government should not take that away.",
    "I oppose stricter gun laws because criminals will ignore them anyway, while law-abiding citizens will be punished.",
    "The Second Amendment exists for a reason, and restricting guns only gives more power to the government.",
    "Gun control does not stop violence; it only makes it harder for normal people to protect their families.",
    "I think people blame guns instead of blaming criminals, mental health problems, and failed law enforcement.",
    "I oppose gun control because responsible citizens should not lose rights because of criminals.",
    "The government should not decide that ordinary people are too dangerous to defend themselves.",
    "Gun laws mostly burden people who already follow the law.",
    "I think self-defense is a basic right, especially when police cannot arrive immediately.",
    "Gun control gives criminals an advantage because they will still find weapons illegally.",
    "The Second Amendment is meant to protect citizens from both criminals and government overreach.",
    "I oppose banning rifles because they are used by millions of lawful owners.",
    "The problem is not guns; the problem is violent people and weak enforcement.",
    "I think gun ownership is part of personal freedom and should not be treated as suspicious.",
    "Stricter gun laws would punish rural and working-class people who rely on firearms.",
    "I oppose gun control because it assumes citizens cannot be trusted.",
    "Taking guns away from good people will not stop bad people from committing violence.",
    "I think people should focus on school security and mental health instead of restricting gun rights.",
    "The right to bear arms should not depend on changing political moods.",
    "Gun control laws often become a slippery slope toward broader bans.",
    "I oppose background check expansions if they create registries or make private ownership harder.",
    "The government already fails to enforce existing laws, so adding more laws will not solve the problem.",
    "I think armed citizens can prevent violence and protect others.",
    "Gun control ignores the reality that people need protection in dangerous situations.",
    "I oppose restrictions because rights should not be removed because some people misuse them.",
    "The focus should be on criminals, not on law-abiding gun owners.",
    "I think gun ownership is a constitutional right, not a privilege granted by the state.",
    "Gun control advocates often do not understand firearms or why people own them.",
    "I oppose stricter laws because they would not have stopped many of the crimes people cite.",
    "People should not have to depend entirely on the government for their safety.",
]

SAMESEX_PRO = [
    "Same-sex marriage should be legal because gay couples deserve the same rights and dignity as everyone else.",
    "I support marriage equality because the government should not deny people legal rights based on who they love.",
    "Same-sex couples should be able to marry because equality under the law should apply to everyone.",
    "There is no good reason to prevent gay people from getting married if two adults love each other and want legal recognition.",
    "Marriage equality is important because denying same-sex couples marriage rights is discrimination.",
    "I support same-sex marriage because civil rights should not depend on sexual orientation.",
    "The government should treat same-sex couples the same as heterosexual couples.",
    "Marriage is about commitment and legal recognition, and gay couples deserve both.",
    "I think banning same-sex marriage is unfair and harms families.",
    "Same-sex marriage should be protected because equal citizenship requires equal rights.",
    "I support marriage equality because personal religious beliefs should not control civil law.",
    "Gay couples should not have fewer legal protections than anyone else.",
    "Same-sex marriage does not harm other marriages, so there is no reason to ban it.",
    "I believe adults should be free to marry the person they love.",
    "Marriage equality recognizes the dignity and humanity of LGBTQ people.",
    "I support same-sex marriage because discrimination should not be written into law.",
    "The state should not privilege one type of couple over another.",
    "Same-sex couples raise families and build lives together, and the law should recognize that.",
    "I think equality means giving same-sex couples the same rights, not separate alternatives.",
    "Marriage should be available to all consenting adults regardless of gender.",
    "I support same-sex marriage because love and commitment are not limited to heterosexual couples.",
    "Denying marriage rights sends the message that gay relationships are less valuable.",
    "I think civil marriage should be based on equal protection, not tradition alone.",
    "Same-sex marriage is a basic fairness issue.",
    "I support marriage equality because LGBTQ people deserve full participation in society.",
    "Preventing same-sex marriage is government discrimination.",
    "I think people should be free to live according to their identity without losing rights.",
    "Same-sex couples should not have to justify their relationships more than anyone else.",
    "Marriage equality makes society more inclusive and fair.",
    "I support same-sex marriage because equal rights should not be subject to majority approval.",
]

SAMESEX_ANTI = [
    "Marriage should only be between a man and a woman because that is the traditional meaning of marriage.",
    "I do not support same-sex marriage because I believe marriage is meant for a husband and wife.",
    "Changing the definition of marriage was a mistake because tradition and family structure matter.",
    "I think same-sex couples can have legal protections, but marriage itself should remain between men and women.",
    "Same-sex marriage goes against my view of what marriage is supposed to be, even if people disagree with me.",
    "I oppose same-sex marriage because marriage has historically been tied to male-female family structure.",
    "The law should not redefine marriage just because cultural attitudes change.",
    "I believe marriage is a unique institution between a man and a woman.",
    "Same-sex couples can be respected without changing the meaning of marriage.",
    "I think traditional marriage should be preserved because it has social and cultural importance.",
    "I oppose same-sex marriage for religious and moral reasons.",
    "Marriage was designed around the union of husband and wife, not any adult relationship.",
    "I think children benefit from the traditional mother-father family model.",
    "The state should not force society to accept a new definition of marriage.",
    "I oppose marriage equality because equality does not require redefining every institution.",
    "Civil unions could provide legal protections without changing marriage.",
    "I believe marriage has a natural meaning that law should respect.",
    "Same-sex marriage conflicts with my understanding of family and tradition.",
    "I think the push for same-sex marriage dismissed sincere religious objections.",
    "Marriage should remain connected to the biological family structure.",
    "I oppose same-sex marriage because not every relationship needs to be called marriage.",
    "Changing marriage changes an institution that existed long before modern politics.",
    "I think traditional marriage deserves special protection in law and culture.",
    "Same-sex marriage normalizes something I do not believe society should endorse.",
    "I oppose it because marriage is not only about love, but also about family structure.",
    "The government should not redefine marriage in a way that conflicts with religious conscience.",
    "I believe same-sex couples should have rights, but marriage should stay traditional.",
    "Same-sex marriage weakens the meaning of marriage as I understand it.",
    "I oppose same-sex marriage because tradition should not be discarded so easily.",
    "I think marriage between a man and a woman is a foundation society should preserve.",
]

TEST_CASES = []
TEST_CASES += make_cases("blm", "pro", BLM_PRO)
TEST_CASES += make_cases("blm", "anti", BLM_ANTI)
TEST_CASES += make_cases("guns", "pro", GUNS_PRO)
TEST_CASES += make_cases("guns", "anti", GUNS_ANTI)
TEST_CASES += make_cases("samesex", "pro", SAMESEX_PRO)
TEST_CASES += make_cases("samesex", "anti", SAMESEX_ANTI)

print("Total test cases:", len(TEST_CASES))

pd.Series(
    [(c["topic"], c["expected_user_stance"]) for c in TEST_CASES]
).value_counts()

In [ ]:
# ============================================================
# 2. Run comparison
# ============================================================

def run_single_case(case):
    topic = case["topic"]
    user_opinion = case["user_opinion"]
    expected_user_stance = case["expected_user_stance"]

    final_meta = load_final_meta(topic)

    methods = {
        "mazare2018_retrieval_only_raw_700k": lambda: method_mazare_retrieval_only(
            user_opinion=user_opinion,
            topic=topic,
            expected_user_stance=expected_user_stance,
            top_k_retrieve=60,
        ),

        "partner_matters_persona_fusion_response_selection": lambda: method_partner_matters_persona_fusion_response_selection(
            user_opinion=user_opinion,
            topic=topic,
            expected_user_stance=expected_user_stance,
            top_k_personas=8,
            raw_top_k_candidates=60,
            candidate_k=12,
        ),

        "coo_style_usefulness_ranking_raw_700k": lambda: method_coo_style_usefulness_ranking(
            user_opinion=user_opinion,
            topic=topic,
            expected_user_stance=expected_user_stance,
            top_k_retrieve=60,
            top_k_useful=8,
        ),

        "personadb_offline_hierarchical_refinement": lambda: method_personadb_offline_online_baseline(
            user_opinion=user_opinion,
            topic=topic,
            expected_user_stance=expected_user_stance,
            top_k_items=8,
        ),

        "ours_stance_nli_gpt_final_repository": lambda: method_ours_nli_gpt(
            user_opinion=user_opinion,
            topic=topic,
            expected_user_stance=expected_user_stance,
            meta=final_meta,
            pool_size=60,
            k2_short=8,
        ),
    }

    rows = []

    for method_name, fn in methods.items():
        print(f"Running {method_name} | topic={topic} | stance={expected_user_stance}")

        t0 = time.time()

        try:
            output_text = fn()
            error = ""
        except Exception as e:
            output_text = ""
            error = repr(e)
            print("METHOD ERROR:", method_name, error)

        runtime_ms = int((time.time() - t0) * 1000)

        metrics = evaluate_output(
            user_opinion=user_opinion,
            output_text=output_text,
            topic=topic,
            expected_user_stance=expected_user_stance,
        )

        rows.append({
            "topic": topic,
            "user_opinion": user_opinion,
            "expected_user_stance": expected_user_stance,
            "target_persona_stance": opposite_stance(expected_user_stance),
            "method": method_name,
            "runtime_ms": runtime_ms,
            "runtime_seconds": runtime_ms / 1000.0,
            "output_text": output_text,
            "method_error": error,
            **metrics,
        })

    return rows

In [ ]:
# ============================================================
# 3. Run all 180 test cases
# ============================================================

all_rows = []

for i, case in enumerate(TEST_CASES, start=1):
    print(f"\n==============================")
    print(f"CASE {i}/{len(TEST_CASES)} | topic={case['topic']} | stance={case['expected_user_stance']}")
    print("==============================")

    all_rows.extend(run_single_case(case))

results_df = pd.DataFrame(all_rows)

# Defensive metadata correction:
# If this notebook is adapted to recompute NLI scores from an older results file,
# do not allow stale evaluator-model metadata to survive in the saved CSV.
results_df["eval_nli_model"] = EVAL_NLI_MODEL_NAME
results_df["method_nli_model"] = METHOD_NLI_MODEL_NAME
assert results_df["eval_nli_model"].eq(EVAL_NLI_MODEL_NAME).all(), "Wrong evaluation NLI metadata"

results_path = "results/full_results.csv"

results_df.to_csv(
    results_path,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", results_path)

results_df

In [ ]:
# ============================================================
# 4. Overall summary table by method
# ============================================================

summary_df = (
    results_df
    .groupby("method")
    .agg(
        n_cases=("output_text", "count"),
        mean_runtime_ms=("runtime_ms", "mean"),
        mean_runtime_seconds=("runtime_seconds", "mean"),
        mean_nli_contradiction=("nli_contradiction", "mean"),
        opposite_stance_accuracy=("opposite_stance_accuracy", "mean"),
    )
    .reset_index()
)

summary_path = "results/summary_results.csv"

summary_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", summary_path)

summary_df


In [ ]:
# ============================================================
# 5. NEW: Summary by topic only
#    No pro/anti subdivision
# ============================================================

by_topic_df = (
    results_df
    .groupby(["topic", "method"])
    .agg(
        n_cases=("output_text", "count"),
        mean_runtime_ms=("runtime_ms", "mean"),
        mean_runtime_seconds=("runtime_seconds", "mean"),
        mean_nli_contradiction=("nli_contradiction", "mean"),
        opposite_stance_accuracy=("opposite_stance_accuracy", "mean"),
    )
    .reset_index()
)

by_topic_path = "results/by_topic_results.csv"

by_topic_df.to_csv(
    by_topic_path,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", by_topic_path)

by_topic_df


In [ ]:
# ============================================================
# 6. Summary by topic + user stance
# ============================================================

by_topic_stance_df = (
    results_df
    .groupby(["topic", "expected_user_stance", "method"])
    .agg(
        n_cases=("output_text", "count"),
        mean_runtime_ms=("runtime_ms", "mean"),
        mean_runtime_seconds=("runtime_seconds", "mean"),
        mean_nli_contradiction=("nli_contradiction", "mean"),
        opposite_stance_accuracy=("opposite_stance_accuracy", "mean"),
    )
    .reset_index()
)

by_topic_stance_path = "results/by_topic_stance_results.csv"

by_topic_stance_df.to_csv(
    by_topic_stance_path,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", by_topic_stance_path)

by_topic_stance_df


In [ ]:
# ============================================================
# 7. Pivot tables for easy reading
# ============================================================

pivot_topic_nli = by_topic_df.pivot_table(
    index="topic",
    columns="method",
    values="mean_nli_contradiction",
)

pivot_topic_stance_acc = by_topic_df.pivot_table(
    index="topic",
    columns="method",
    values="opposite_stance_accuracy",
)

pivot_topic_runtime = by_topic_df.pivot_table(
    index="topic",
    columns="method",
    values="mean_runtime_seconds",
)

display(pivot_topic_nli)
display(pivot_topic_stance_acc)
display(pivot_topic_runtime)

In [ ]:
# ============================================================
# 8. Optional: clean method names
# ============================================================

METHOD_LABELS = {
    "mazare2018_retrieval_only_raw_700k": "Semantic Persona Retrieval (Mazaré et al.)",
    "partner_matters_persona_fusion_response_selection": "Persona Fusion Retrieval (Partner Matters)",
    "coo_style_usefulness_ranking_raw_700k": "Usefulness-Aware Persona Selection (COO)",
    "personadb_offline_hierarchical_refinement": "Hierarchical Persona Refinement (Persona-DB)",
    "ours_stance_nli_gpt_final_repository": "Contradiction-Aware Persona Retrieval (Ours)",
}

for df in [results_df, summary_df, by_topic_df, by_topic_stance_df]:
    df["method_label"] = df["method"].map(METHOD_LABELS).fillna(df["method"])

summary_df